In [3]:
import csv
import os
from dotenv import load_dotenv
from pymongo import MongoClient

In [ ]:
load_dotenv(dotenv_path=".env.local")

# Get URI
uri = os.getenv("MONGODB_URI")
if not uri:
    raise ValueError("MONGO_URI not found in .env.local")

client = MongoClient(uri)  
db = client['planets']                 
collection = db['planets']            

with open('../data/summary.csv', newline='', encoding='utf-8') as csvfile:
    reader = csv.DictReader(csvfile)
    try:
        first_row = next(reader)  # Get only the first row
        doc = dict(first_row)

        # Insert first row into MongoDB
        result = collection.insert_one(doc)
        print(f"Inserted document with _id: {result.inserted_id}")

    except StopIteration:
        print("CSV file is empty, nothing to insert.")


Inserted document with _id: 6920c060eb47b3fb1163770d


In [23]:
load_dotenv(dotenv_path=".env.local")

# Get URI
uri = os.getenv("MONGODB_URI")
if not uri:
    raise ValueError("MONGO_URI not found in .env.local")

client = MongoClient(uri)  
db = client['planets']                 
collection = db['planets']            

with open('../data/summary.csv', newline='', encoding='utf-8') as csvfile:
    reader = csv.DictReader(csvfile)

    documents = []
    for row in reader: 
        doc = row
        obj = dict(row) 

        # Insert row into MongoDB
        result = collection.insert_one(obj)
        print(f"Inserted document with _id: {result.inserted_id}")



Inserted document with _id: 692231e89e037df691c37529
Inserted document with _id: 692231e99e037df691c3752a
Inserted document with _id: 692231e99e037df691c3752b
Inserted document with _id: 692231e99e037df691c3752c
Inserted document with _id: 692231e99e037df691c3752d
Inserted document with _id: 692231e99e037df691c3752e
Inserted document with _id: 692231e99e037df691c3752f
Inserted document with _id: 692231e99e037df691c37530


In [ ]:
result = collection.update_many(
    {"planet": {"$exists": True}}, 
    {"$rename": { "planet": "name"}}
)

print(f"Modified {result.modified_count} documents")

Modified 0 documents


In [17]:
result = collection.update_many(
    { "gravity_m-s2": {"$exists": True}}, 
    { "$rename": { "gravity_m-s2": "gravity_m_s2"}}
)

print(f"Modified {result.modified_count} documents")

Modified 8 documents


In [18]:
result = collection.update_many(
    { "density_kg-km3": {"$exists": True}}, 
    { "$rename": { "density_kg-km3": "density_kg_km3"}}
)

print(f"Modified {result.modified_count} documents")

Modified 8 documents


In [21]:
result = collection.update_many(
    { "planet": {"$exists": True}}, 
    { "$rename": { "planet": "name"}}
)

print(f"Modified {result.modified_count} documents")

Modified 0 documents


In [26]:
fields = {
    "high_temp_c":"$toDouble", 
    "low_temp_c":"$toDouble", 
    "atmos_pressure_mbar":"$toDouble",
    "atmos_N":"$toDouble",
    "atmos_O":"$toDouble",
    "atmos_CO2":"$toDouble",
    "atmos_CH4":"$toDouble",
    "atmos_H":"$toDouble",
    "gravity_m_s2":"$toDouble"
}

set_stage = {
    field: {op: f"${field}"}
    for field, op in fields.items()
}

result = collection.update_many(
    {}, 
    [
        {"$set": set_stage}
    ]
)

print(f"Modified {result.modified_count} documents")

Modified 8 documents


In [ ]:
# Load environment variables from .env.local
load_dotenv()

MONGO_URI = os.getenv("MONGODB_URI")  # Your MongoDB URI
DB_NAME = "planets"
COLLECTION_NAME = "planets"
CSV_FILE = "../data/earth_adjusted_summary.csv"  # Path to your Earth-adjusted CSV

# Connect to MongoDB
client = MongoClient(MONGO_URI)
db = client[DB_NAME]
collection = db[COLLECTION_NAME]

# Open the CSV and iterate over each row
with open(CSV_FILE, newline='', encoding='utf-8') as csvfile:
    reader = csv.DictReader(csvfile)
    
    for row in reader:
        planet_name = row.get("planet").strip()  # remove any leading/trailing spaces
        if not planet_name:
            continue

        # Prepare Earth-adjusted fields
        earth_adjusted_fields = {
            "mass_kg_earthAdjusted": float(row["mass_kg"]),
            "volume_km3_earthAdjusted": float(row["volume_km3"]),
            "density_kg_km3_earthAdjusted": float(row["density_kg_km3"]),
            "gravity_m_s2_earthAdjusted": float(row["gravity_m_s2"]),
            "high_temp_c_earthAdjusted": float(row["high_temp_c"]),
            "low_temp_c_earthAdjusted": float(row["low_temp_c"]),
            "atmos_pressure_mbar_earthAdjusted": float(row["atmos_pressure_mbar"]),
            "atmos_N_earthAdjusted": float(row["atmos_N"]),
            "atmos_O_earthAdjusted": float(row["atmos_O"]),
            "atmos_CO2_earthAdjusted": float(row["atmos_CO2"]),
            "atmos_CH4_earthAdjusted": float(row["atmos_CH4"]),
            "atmos_H_earthAdjusted": float(row["atmos_H"]),
        }

        # Update the document in MongoDB
        result = collection.update_one(
            {"name": planet_name},  # match by the 'name' field
            {"$set": earth_adjusted_fields}
        )

        if result.matched_count == 0:
            print(f"Warning: no document found for planet '{planet_name}'")
        else:
            print(f"Updated planet '{planet_name}' with Earth-adjusted values")


Updated planet 'Mercury' with Earth-adjusted values
Updated planet 'Venus' with Earth-adjusted values
Updated planet 'Earth' with Earth-adjusted values
Updated planet 'Mars' with Earth-adjusted values
Updated planet 'Jupiter' with Earth-adjusted values
Updated planet 'Neptune' with Earth-adjusted values


In [31]:
# Load environment variables from .env.local
load_dotenv()

MONGO_URI = os.getenv("MONGODB_URI")  # Your MongoDB URI
DB_NAME = "planets"
COLLECTION_NAME = "planets"
CSV_FILE = "../data/earth_adjusted_summary.csv"  # Path to your Earth-adjusted CSV

# Connect to MongoDB
client = MongoClient(MONGO_URI)
db = client[DB_NAME]
collection = db[COLLECTION_NAME]

# Open the CSV and iterate over each row
with open(CSV_FILE, newline='', encoding='utf-8') as csvfile:
    reader = csv.DictReader(csvfile)
    
    for row in reader:
        planet_name = row.get("planet")  # remove any leading/trailing spaces
        if not planet_name:
            continue

        if planet_name == "Saturn": 
            planet_name = "Saturn "

        if planet_name == "Uranus": 
            planet_name = "Uranus "

        # Prepare Earth-adjusted fields
        earth_adjusted_fields = {
            "mass_kg_earthAdjusted": float(row["mass_kg"]),
            "volume_km3_earthAdjusted": float(row["volume_km3"]),
            "density_kg_km3_earthAdjusted": float(row["density_kg_km3"]),
            "gravity_m_s2_earthAdjusted": float(row["gravity_m_s2"]),
            "high_temp_c_earthAdjusted": float(row["high_temp_c"]),
            "low_temp_c_earthAdjusted": float(row["low_temp_c"]),
            "atmos_pressure_mbar_earthAdjusted": float(row["atmos_pressure_mbar"]),
            "atmos_N_earthAdjusted": float(row["atmos_N"]),
            "atmos_O_earthAdjusted": float(row["atmos_O"]),
            "atmos_CO2_earthAdjusted": float(row["atmos_CO2"]),
            "atmos_CH4_earthAdjusted": float(row["atmos_CH4"]),
            "atmos_H_earthAdjusted": float(row["atmos_H"]),
        }

        # Update the document in MongoDB
        result = collection.update_one(
            {"name": planet_name},  # match by the 'name' field
            {"$set": earth_adjusted_fields}
        )

        if result.matched_count == 0:
            print(f"Warning: no document found for planet '{planet_name}'")
        else:
            print(f"Updated planet '{planet_name}' with Earth-adjusted values")


Updated planet 'Mercury' with Earth-adjusted values
Updated planet 'Venus' with Earth-adjusted values
Updated planet 'Earth' with Earth-adjusted values
Updated planet 'Mars' with Earth-adjusted values
Updated planet 'Jupiter' with Earth-adjusted values
Updated planet 'Saturn ' with Earth-adjusted values
Updated planet 'Uranus ' with Earth-adjusted values
Updated planet 'Neptune' with Earth-adjusted values


In [1]:
import math
from typing import Dict, List

# Earth reference (from your CSV — mass=1, volume=1, etc.)
EARTH = {
    "mass_kg": 1.0,
    "volume_km3": 1.0,
    "density_kg_km3": 1.0,
    "gravity_m_s2": 1.0,
    "high_temp_c": 1.0,
    "low_temp_c": 1.0,
    "atmos_pressure_mbar": 1.0,
    "atmos_vec": [1.0, 1.0, 1.0, 1.0, 1.0],  # N, O, CO2, CH4, H (from CSV Earth row)
}

WEIGHTS = {
    "mass": 0.15,
    "volume": 0.05,
    "density": 0.10,
    "gravity": 0.15,
    "high_temp": 0.12,
    "low_temp": 0.08,
    "pressure": 0.10,
    "composition": 0.25,
}

def safe_float(x):
    try:
        return float(x)
    except Exception:
        return None

def normalize_composition(row: Dict[str, str]) -> List[float]:
    keys = ["atmos_N", "atmos_O", "atmos_CO2", "atmos_CH4", "atmos_H"]
    vals = [safe_float(row.get(k, 0) or 0) for k in keys]
    s = sum(vals)
    if s == 0:
        # fallback: treat as all zeros (max distance)
        return [0.0]*5
    return [v / s for v in vals]

def cosine_similarity(a: List[float], b: List[float]) -> float:
    dot = sum(x*y for x,y in zip(a,b))
    norma = math.sqrt(sum(x*x for x in a))
    normb = math.sqrt(sum(y*y for y in b))
    if norma == 0 or normb == 0:
        return 0.0
    return dot / (norma * normb)

def clamp01(x: float) -> float:
    return max(0.0, min(1.0, x))

def earth_similarity(row: Dict[str, str]) -> float:
    # parse numeric features
    mass = safe_float(row.get("mass_kg"))
    volume = safe_float(row.get("volume_km3"))
    density = safe_float(row.get("density_kg_km3"))
    gravity = safe_float(row.get("gravity_m_s2"))
    high_temp = safe_float(row.get("high_temp_c"))
    low_temp = safe_float(row.get("low_temp_c"))
    pressure = safe_float(row.get("atmos_pressure_mbar"))

    # distances per feature (0 = identical)
    parts = {}
    # mass & volume: log-ratio normalized by an expected max ratio (e.g., 1000)
    MAX_LOG_RATIO = math.log10(1000)  # tune as needed
    if mass is not None and EARTH["mass_kg"] > 0:
        parts["mass"] = clamp01(abs(math.log10(mass / EARTH["mass_kg"])) / MAX_LOG_RATIO)
    else:
        parts["mass"] = None

    if volume is not None and EARTH["volume_km3"] > 0:
        parts["volume"] = clamp01(abs(math.log10(volume / EARTH["volume_km3"])) / MAX_LOG_RATIO)
    else:
        parts["volume"] = None

    # linear ratio differences
    def ratio_dist(val, earth_val, cap=10.0):
        if val is None or earth_val is None or earth_val == 0:
            return None
        r = abs(val / earth_val - 1.0)
        return clamp01(r / cap)  # cap scales differences (tune cap)

    parts["density"] = ratio_dist(density, EARTH["density_kg_km3"], cap=3.0)
    parts["gravity"] = ratio_dist(gravity, EARTH["gravity_m_s2"], cap=3.0)
    parts["high_temp"] = ratio_dist(high_temp, EARTH["high_temp_c"], cap=50.0)
    parts["low_temp"] = ratio_dist(low_temp, EARTH["low_temp_c"], cap=50.0)
    parts["pressure"] = ratio_dist(pressure, EARTH["atmos_pressure_mbar"], cap=10.0)

    # composition: cosine distance between normalized vectors
    comp_vec = normalize_composition(row)
    earth_comp = normalize_composition({
        "atmos_N": EARTH["atmos_vec"][0],
        "atmos_O": EARTH["atmos_vec"][1],
        "atmos_CO2": EARTH["atmos_vec"][2],
        "atmos_CH4": EARTH["atmos_vec"][3],
        "atmos_H": EARTH["atmos_vec"][4],
    })
    cos_sim = cosine_similarity(comp_vec, earth_comp)
    parts["composition"] = clamp01(1.0 - cos_sim)  # 0 when identical, 1 when orthogonal

    # build weighted average over available parts
    total_weight = 0.0
    weighted_distance = 0.0
    for key, weight in WEIGHTS.items():
        val = parts.get(key)
        if val is None:
            continue
        weighted_distance += weight * val
        total_weight += weight

    if total_weight == 0:
        return 0.0

    # normalize distance to 0..1 and convert to 0..100 similarity
    normalized_distance = weighted_distance / total_weight
    similarity = clamp01(1.0 - normalized_distance) * 100.0
    return round(similarity, 2)


In [14]:
with open('../data/earth_adjusted_summary.csv', newline='', encoding='utf-8') as csvfile:
    reader = csv.DictReader(csvfile)

    results = {}
    for row in reader: 
        planet = row
        name = planet['planet']
        similarity = earth_similarity(row) 

        results[name] = similarity

    
print(results)


{'Mercury': 72.23, 'Venus': 82.34, 'Earth': 100.0, 'Mars': 74.77, 'Jupiter': 58.22, 'Saturn': 67.24, 'Uranus': 72.97, 'Neptune': 71.87}


In [ ]:
results['Mercury'] = results['Mercury'] - 2.00
results['Venus'] = results['Venus'] - 18
results['Mars'] = results['Mars'] + 7
results['Uranus'] = results['Uranus'] - 4
results['Neptune'] = results['Neptune'] - 4

In [16]:
print(results)

{'Mercury': 70.23, 'Venus': 64.34, 'Earth': 100.0, 'Mars': 81.77, 'Jupiter': 58.22, 'Saturn': 67.24, 'Uranus': 68.97, 'Neptune': 67.87}


In [18]:
load_dotenv(dotenv_path=".env.local")

# Get URI
uri = os.getenv("MONGODB_URI")
if not uri:
    raise ValueError("MONGO_URI not found in .env.local")

client = MongoClient(uri)  
db = client['planets']                 
collection = db['planets']  

for planet_name, score in results.items():
    if planet_name == "Saturn": 
        planet_name = "Saturn "

    if planet_name == "Uranus": 
        planet_name = "Uranus "

    result = collection.update_one(
        {"name": planet_name},
        {
            "$set": {
                "earth_similarity_score": score
            }
        }
    )

    print(f"{planet_name}: matched={result.matched_count}, modified={result.modified_count}")



Mercury: matched=1, modified=0
Venus: matched=1, modified=0
Earth: matched=1, modified=0
Mars: matched=1, modified=0
Jupiter: matched=1, modified=0
Saturn : matched=1, modified=1
Uranus : matched=1, modified=1
Neptune: matched=1, modified=0


In [20]:
descriptions = {
    'Mercury': "The closest planet to our sun: lacking any atmosphere, Mercury experiences wild temperature swings.",
    'Venus': "Venus is the hottest planet in our solar system with a thick, toxic atmosphere.",
    'Earth': "Earth. The home of the only known life in the solar system, galaxy, and universe.",
    'Mars': "Mars, the red planet, known for its iron oxide surface and potential for future human colonization.",
    'Jupiter': "The largest planet in our solar system, Jupiters gassy surface is dominated by storms.", 
    'Saturn ': "Known for its iconic icy rings, Saturn is a giant world with a deep atmosphere and dozens of moons.", 
    'Uranus ': "An ice giant tipped dramatically on its side, Uranus has a pale blue atmosphere and extreme seasonal cycles.", 
    'Neptune': "Neptune is the most remote planet in our solar system and is characterized by fierce frozen winds."
}

In [22]:
for planet_name, description in descriptions.items():
    if planet_name == "Saturn": 
        planet_name = "Saturn "

    if planet_name == "Uranus": 
        planet_name = "Uranus "

    result = collection.update_one(
        {"name": planet_name},
        {
            "$set": {
                "description": description
            }
        }
    )

    print(f"{planet_name}: matched={result.matched_count}, modified={result.modified_count}")

Mercury: matched=1, modified=1
Venus: matched=1, modified=1
Earth: matched=1, modified=1
Mars: matched=1, modified=1
Jupiter: matched=1, modified=1
Saturn : matched=1, modified=1
Uranus : matched=1, modified=1
Neptune: matched=1, modified=1
